In [0]:
# ===================================================
# BLOCK 1 — DEPENDENCY CHECKS (PYTHON)
# ===================================================

"""
Confirm that every validated Silver source and Gold dimension required by
the analytical model is available before replacing fact and mart tables.
"""

required_tables = [
    "semiconplus_portfolio.silver.production_lots",
    "semiconplus_portfolio.silver.unit_test_results",
    "semiconplus_portfolio.silver.equipment_events",
    "semiconplus_portfolio.silver.tester_logs",
    "semiconplus_portfolio.gold.dim_date",
    "semiconplus_portfolio.gold.dim_site",
    "semiconplus_portfolio.gold.dim_product_group",
    "semiconplus_portfolio.gold.dim_device_scd2",
    "semiconplus_portfolio.gold.dim_equipment",
    "semiconplus_portfolio.gold.dim_defect",
    "semiconplus_portfolio.gold.dim_test_program",
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

assert not missing_tables, f"Required tables are missing: {missing_tables}"

print("Gold fact and mart dependencies passed.")

In [0]:
%sql
-- # ===================================================
-- # BLOCK 2 — LOT-PERFORMANCE FACT (SQL)
-- # ===================================================

-- One row per accepted production lot.
-- Device history is resolved using the lot production date.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.fact_lot_performance
USING DELTA
COMMENT 'Lot-grain manufacturing quantities and yield with conformed keys'
AS
WITH resolved AS (
    SELECT
        XXHASH64(l.lot_id) AS lot_key,
        l.lot_id,
        COALESCE(d.date_key, 0) AS production_date_key,
        COALESCE(s.site_key, 0) AS site_key,
        COALESCE(pg.product_group_key, 0) AS product_group_key,
        COALESCE(dv.device_key, 0) AS device_key,
        COALESCE(eq.equipment_key, 0) AS equipment_key,
        l.production_date,
        l.start_timestamp_utc,
        l.quantity_started,
        l.quantity_passed,
        l.quantity_failed,
        l.source_actual_yield,
        l.calculated_yield AS manufacturing_yield,
        l.test_program_revision,
        l.source_system,
        l._source_file_path,
        l._bronze_pipeline_run_id,
        l._silver_pipeline_run_id,
        CURRENT_TIMESTAMP() AS _gold_processed_at_utc
    FROM semiconplus_portfolio.silver.production_lots l
    LEFT JOIN semiconplus_portfolio.gold.dim_date d
        ON d.calendar_date = l.production_date
    LEFT JOIN semiconplus_portfolio.gold.dim_site s
        ON s.site_id = l.site_id
    LEFT JOIN semiconplus_portfolio.gold.dim_product_group pg
        ON pg.product_group_id = l.product_group_id
    LEFT JOIN semiconplus_portfolio.gold.dim_device_scd2 dv
        ON dv.device_id = l.device_id
       AND l.production_date BETWEEN dv.effective_from AND dv.effective_to
    LEFT JOIN semiconplus_portfolio.gold.dim_equipment eq
        ON eq.equipment_id = l.equipment_id
)
SELECT * FROM resolved;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 3 — UNIT-TEST-RESULT FACT (SQL)
-- # ===================================================

-- One row per accepted tested unit result.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.fact_unit_test_results
USING DELTA
COMMENT 'Unit-test-result grain with defect, program, and conformed keys'
AS
SELECT
    XXHASH64(t.test_result_id) AS test_result_key,
    t.test_result_id,
    XXHASH64(t.lot_id) AS lot_key,
    COALESCE(dd.date_key, 0) AS event_date_key,
    COALESCE(ds.site_key, 0) AS site_key,
    COALESCE(dpg.product_group_key, 0) AS product_group_key,
    COALESCE(dv.device_key, 0) AS device_key,
    COALESCE(de.equipment_key, 0) AS equipment_key,
    COALESCE(df.defect_key, 0) AS defect_key,
    COALESCE(tp.test_program_key, 0) AS test_program_key,
    t.event_timestamp_utc,
    t.lot_id,
    t.unit_sequence,
    t.test_status,
    t.defect_code,
    t.test_time_seconds,
    t.program_revision,
    CASE WHEN t.test_status = 'PASS' THEN 1 ELSE 0 END AS passed_unit_count,
    CASE WHEN t.test_status = 'FAIL' THEN 1 ELSE 0 END AS failed_unit_count,
    1 AS tested_unit_count,
    t._source_file_path,
    t._bronze_pipeline_run_id,
    t._silver_pipeline_run_id,
    CURRENT_TIMESTAMP() AS _gold_processed_at_utc
FROM semiconplus_portfolio.silver.unit_test_results t
LEFT JOIN semiconplus_portfolio.gold.dim_date dd
    ON dd.calendar_date = CAST(t.event_timestamp_utc AS DATE)
LEFT JOIN semiconplus_portfolio.gold.dim_site ds
    ON ds.site_id = t.site_id
LEFT JOIN semiconplus_portfolio.gold.dim_product_group dpg
    ON dpg.product_group_id = t.product_group_id
LEFT JOIN semiconplus_portfolio.gold.dim_device_scd2 dv
    ON dv.device_id = t.device_id
   AND CAST(t.event_timestamp_utc AS DATE)
       BETWEEN dv.effective_from AND dv.effective_to
LEFT JOIN semiconplus_portfolio.gold.dim_equipment de
    ON de.equipment_id = t.equipment_id
LEFT JOIN semiconplus_portfolio.gold.dim_defect df
    ON df.defect_code = t.defect_code
LEFT JOIN semiconplus_portfolio.gold.dim_test_program tp
    ON tp.device_id = t.device_id
   AND tp.program_revision = t.program_revision;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 4 — EQUIPMENT-EVENT FACT (SQL)
-- # ===================================================

-- One row per accepted equipment event.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.fact_equipment_events
USING DELTA
COMMENT 'Equipment-event grain with duration and loss classification'
AS
SELECT
    XXHASH64(e.event_id) AS equipment_event_key,
    e.event_id,
    COALESCE(dd.date_key, 0) AS event_date_key,
    COALESCE(ds.site_key, 0) AS site_key,
    COALESCE(de.equipment_key, 0) AS equipment_key,
    e.event_timestamp_utc,
    e.event_type,
    e.duration_seconds,
    e.alarm_code,
    e.source_system,
    CASE WHEN e.event_type = 'RUN' THEN e.duration_seconds ELSE 0 END
        AS run_seconds,
    CASE WHEN e.event_type = 'PLANNED_DOWNTIME'
         THEN e.duration_seconds ELSE 0 END AS planned_downtime_seconds,
    CASE WHEN e.event_type IN (
            'UNPLANNED_DOWNTIME', 'ALARM', 'MAINTENANCE'
         ) THEN e.duration_seconds ELSE 0 END AS unplanned_loss_seconds,
    CASE WHEN e.event_type IN ('IDLE', 'SETUP')
         THEN e.duration_seconds ELSE 0 END AS operational_loss_seconds,
    e._source_file_path,
    e._bronze_pipeline_run_id,
    e._silver_pipeline_run_id,
    CURRENT_TIMESTAMP() AS _gold_processed_at_utc
FROM semiconplus_portfolio.silver.equipment_events e
LEFT JOIN semiconplus_portfolio.gold.dim_date dd
    ON dd.calendar_date = CAST(e.event_timestamp_utc AS DATE)
LEFT JOIN semiconplus_portfolio.gold.dim_site ds
    ON ds.site_id = e.site_id
LEFT JOIN semiconplus_portfolio.gold.dim_equipment de
    ON de.equipment_id = e.equipment_id;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 5 — DATA-QUALITY FACT (SQL)
-- # ===================================================

-- One row per recorded dataset validation execution.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.fact_data_quality
USING DELTA
COMMENT 'Dataset-level validation results from pipeline monitoring history'
AS
SELECT
    XXHASH64(
        pipeline_run_id,
        dataset_name,
        CAST(validated_at_utc AS STRING)
    ) AS data_quality_key,
    pipeline_run_id,
    dataset_name,
    source_row_count,
    accepted_row_count,
    rejected_row_count,
    CASE
        WHEN source_row_count = 0 THEN CAST(0 AS DECIMAL(9,6))
        ELSE CAST(accepted_row_count / source_row_count AS DECIMAL(9,6))
    END AS acceptance_rate,
    validation_status,
    validated_at_utc,
    COALESCE(dd.date_key, 0) AS validation_date_key,
    CURRENT_TIMESTAMP() AS _gold_processed_at_utc
FROM semiconplus_portfolio.monitoring.data_quality_results q
LEFT JOIN semiconplus_portfolio.gold.dim_date dd
    ON dd.calendar_date = CAST(q.validated_at_utc AS DATE);

In [0]:
%sql
-- # ===================================================
-- # BLOCK 6 — DAILY YIELD MART (SQL)
-- # ===================================================

-- Quantity-weighted manufacturing yield by date and site.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_daily_yield
USING DELTA
COMMENT 'Daily site-level manufacturing yield for Power BI'
AS
SELECT
    f.production_date_key,
    f.site_key,
    COUNT(*) AS lot_count,
    SUM(f.quantity_started) AS quantity_started,
    SUM(f.quantity_passed) AS quantity_passed,
    SUM(f.quantity_failed) AS quantity_failed,
    CAST(
        SUM(f.quantity_passed) / NULLIF(SUM(f.quantity_started), 0)
        AS DECIMAL(9,6)
    ) AS manufacturing_yield,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM semiconplus_portfolio.gold.fact_lot_performance f
GROUP BY f.production_date_key, f.site_key;


In [0]:
%sql
-- # ===================================================
-- # BLOCK 7 — DEVICE YIELD MART (SQL)
-- # ===================================================

-- Quantity-weighted yield by date, device, product group, and site.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_device_yield
USING DELTA
COMMENT 'Daily device manufacturing yield and target comparison'
AS
SELECT
    f.production_date_key,
    f.site_key,
    f.product_group_key,
    f.device_key,
    COUNT(*) AS lot_count,
    SUM(f.quantity_started) AS quantity_started,
    SUM(f.quantity_passed) AS quantity_passed,
    SUM(f.quantity_failed) AS quantity_failed,
    CAST(
        SUM(f.quantity_passed) / NULLIF(SUM(f.quantity_started), 0)
        AS DECIMAL(9,6)
    ) AS manufacturing_yield,
    MAX(d.target_yield) AS target_yield,
    CAST(
        SUM(f.quantity_passed) / NULLIF(SUM(f.quantity_started), 0)
        - MAX(d.target_yield)
        AS DECIMAL(9,6)
    ) AS yield_gap_to_target,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM semiconplus_portfolio.gold.fact_lot_performance f
LEFT JOIN semiconplus_portfolio.gold.dim_device_scd2 d
    ON d.device_key = f.device_key
GROUP BY
    f.production_date_key,
    f.site_key,
    f.product_group_key,
    f.device_key;


In [0]:
%sql
-- # ===================================================
-- # BLOCK 8 — DEFECT PARETO MART (SQL)
-- # ===================================================

-- Daily defect counts, percentages, and cumulative Pareto contribution.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_defect_pareto
USING DELTA
COMMENT 'Daily defect Pareto with rank and cumulative contribution'
AS
WITH defect_counts AS (
    SELECT
        event_date_key,
        site_key,
        device_key,
        defect_key,
        COUNT(*) AS defect_count
    FROM semiconplus_portfolio.gold.fact_unit_test_results
    WHERE failed_unit_count = 1
    GROUP BY event_date_key, site_key, device_key, defect_key
),
ranked AS (
    SELECT
        *,
        SUM(defect_count) OVER (
            PARTITION BY event_date_key, site_key, device_key
        ) AS total_defect_count,
        DENSE_RANK() OVER (
            PARTITION BY event_date_key, site_key, device_key
            ORDER BY defect_count DESC, defect_key
        ) AS defect_rank,
        SUM(defect_count) OVER (
            PARTITION BY event_date_key, site_key, device_key
            ORDER BY defect_count DESC, defect_key
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_defect_count
    FROM defect_counts
)
SELECT
    *,
    CAST(defect_count / NULLIF(total_defect_count, 0) AS DECIMAL(9,6))
        AS defect_percentage,
    CAST(cumulative_defect_count / NULLIF(total_defect_count, 0)
        AS DECIMAL(9,6)) AS cumulative_defect_percentage,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM ranked;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 9 — EQUIPMENT OEE MART (SQL)
-- # ===================================================

-- OEE components by equipment and day.
-- Availability uses scheduled seconds after planned downtime.
-- Performance compares processed units with rated output during run time.
-- Quality uses passed divided by started units from tester logs.

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_equipment_oee_daily
USING DELTA
COMMENT 'Daily equipment Availability, Performance, Quality, and OEE'
AS
WITH event_time AS (
    SELECT
        event_date_key,
        equipment_key,
        site_key,
        SUM(duration_seconds) AS total_event_seconds,
        SUM(run_seconds) AS run_seconds,
        SUM(planned_downtime_seconds) AS planned_downtime_seconds,
        SUM(unplanned_loss_seconds) AS unplanned_loss_seconds,
        SUM(operational_loss_seconds) AS operational_loss_seconds
    FROM semiconplus_portfolio.gold.fact_equipment_events
    GROUP BY event_date_key, equipment_key, site_key
),
production AS (
    SELECT
        CAST(DATE_FORMAT(event_timestamp_utc, 'yyyyMMdd') AS INT)
            AS event_date_key,
        de.equipment_key,
        SUM(quantity_started) AS quantity_started,
        SUM(quantity_passed) AS quantity_passed,
        SUM(quantity_failed) AS quantity_failed
    FROM semiconplus_portfolio.silver.tester_logs l
    LEFT JOIN semiconplus_portfolio.gold.dim_equipment de
        ON de.equipment_id = l.equipment_id
    GROUP BY
        CAST(DATE_FORMAT(event_timestamp_utc, 'yyyyMMdd') AS INT),
        de.equipment_key
),
components AS (
    SELECT
        e.event_date_key,
        e.site_key,
        e.equipment_key,
        e.total_event_seconds,
        e.run_seconds,
        e.planned_downtime_seconds,
        e.unplanned_loss_seconds,
        e.operational_loss_seconds,
        COALESCE(p.quantity_started, 0) AS quantity_started,
        COALESCE(p.quantity_passed, 0) AS quantity_passed,
        COALESCE(p.quantity_failed, 0) AS quantity_failed,
        de.rated_units_per_hour,
        CAST(
            e.run_seconds /
            NULLIF(e.total_event_seconds - e.planned_downtime_seconds, 0)
            AS DECIMAL(9,6)
        ) AS availability,
        CAST(
            LEAST(
                1.0,
                COALESCE(p.quantity_started, 0) /
                NULLIF((e.run_seconds / 3600.0) * de.rated_units_per_hour, 0)
            ) AS DECIMAL(9,6)
        ) AS performance,
        CAST(
            COALESCE(p.quantity_passed, 0) /
            NULLIF(COALESCE(p.quantity_started, 0), 0)
            AS DECIMAL(9,6)
        ) AS quality
    FROM event_time e
    LEFT JOIN production p
        ON p.event_date_key = e.event_date_key
       AND p.equipment_key = e.equipment_key
    LEFT JOIN semiconplus_portfolio.gold.dim_equipment de
        ON de.equipment_key = e.equipment_key
)
SELECT
    *,
    CAST(availability * performance * quality AS DECIMAL(9,6)) AS oee,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM components;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 10 — EQUIPMENT LOSSES MART (SQL) 
-- # ===================================================

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_equipment_losses
USING DELTA
COMMENT 'Equipment loss duration by date, site, equipment, and event type'
AS
SELECT
    event_date_key,
    site_key,
    equipment_key,
    event_type,
    COUNT(*) AS event_count,
    SUM(duration_seconds) AS loss_seconds,
    CAST(SUM(duration_seconds) / 60.0 AS DECIMAL(18,2)) AS loss_minutes,
    CAST(SUM(duration_seconds) / 3600.0 AS DECIMAL(18,2)) AS loss_hours,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM semiconplus_portfolio.gold.fact_equipment_events
WHERE event_type <> 'RUN'
GROUP BY event_date_key, site_key, equipment_key, event_type;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 11 — LOT TRACEABILITY MART (SQL)
-- # ===================================================

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_lot_traceability
USING DELTA
COMMENT 'Lot-level production and unit-test traceability for investigation'
AS
SELECT
    l.lot_key,
    l.lot_id,
    l.production_date_key,
    l.site_key,
    l.product_group_key,
    l.device_key,
    l.equipment_key,
    l.quantity_started,
    l.quantity_passed,
    l.quantity_failed,
    l.manufacturing_yield,
    COUNT(t.test_result_key) AS unit_test_result_count,
    SUM(COALESCE(t.passed_unit_count, 0)) AS unit_pass_count,
    SUM(COALESCE(t.failed_unit_count, 0)) AS unit_fail_count,
    CAST(
        SUM(COALESCE(t.passed_unit_count, 0)) /
        NULLIF(COUNT(t.test_result_key), 0)
        AS DECIMAL(9,6)
    ) AS unit_test_pass_rate,
    AVG(t.test_time_seconds) AS average_test_time_seconds,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM semiconplus_portfolio.gold.fact_lot_performance l
LEFT JOIN semiconplus_portfolio.gold.fact_unit_test_results t
    ON t.lot_key = l.lot_key
GROUP BY
    l.lot_key,
    l.lot_id,
    l.production_date_key,
    l.site_key,
    l.product_group_key,
    l.device_key,
    l.equipment_key,
    l.quantity_started,
    l.quantity_passed,
    l.quantity_failed,
    l.manufacturing_yield;

In [0]:
%sql
-- # ===================================================
-- # BLOCK 12 — DATA-QUALITY SUMMARY MART (SQL)
-- # ===================================================

CREATE OR REPLACE TABLE semiconplus_portfolio.gold.mart_data_quality_summary
USING DELTA
COMMENT 'Latest dataset-level validation result for operational reporting'
AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY dataset_name
            ORDER BY validated_at_utc DESC, data_quality_key DESC
        ) AS result_rank
    FROM semiconplus_portfolio.gold.fact_data_quality
)
SELECT
    dataset_name,
    pipeline_run_id,
    source_row_count,
    accepted_row_count,
    rejected_row_count,
    acceptance_rate,
    validation_status,
    validated_at_utc,
    validation_date_key,
    CURRENT_TIMESTAMP() AS refreshed_at_utc
FROM ranked
WHERE result_rank = 1;

In [0]:
# ===================================================
# BLOCK 13 — FACT AND MART SUMMARY (PYTHON)
# ===================================================

"""
Publish persisted Gold populations before the separate validation notebook
runs independent reconciliation controls.
"""

gold_tables = [
    "semiconplus_portfolio.gold.fact_lot_performance",
    "semiconplus_portfolio.gold.fact_unit_test_results",
    "semiconplus_portfolio.gold.fact_equipment_events",
    "semiconplus_portfolio.gold.fact_data_quality",
    "semiconplus_portfolio.gold.mart_daily_yield",
    "semiconplus_portfolio.gold.mart_device_yield",
    "semiconplus_portfolio.gold.mart_defect_pareto",
    "semiconplus_portfolio.gold.mart_equipment_oee_daily",
    "semiconplus_portfolio.gold.mart_equipment_losses",
    "semiconplus_portfolio.gold.mart_lot_traceability",
    "semiconplus_portfolio.gold.mart_data_quality_summary",
]

for table_name in gold_tables:
    print(f"{table_name}: {spark.table(table_name).count():,}")

print("GOLD FACT AND MART BUILD COMPLETED")